# Formula/Table Crop OCR Hard Sample Finder

This notebook runs Qwen3-VL crop OCR on the already-built `formula` and `table` crops, compares each prediction with `labels.jsonl`, and exports the samples whose normalized CER is greater than zero.

Inputs:
- `ocr_region_crops_hybrid/labels.jsonl`
- `ocr_region_crops_hybrid/formula/*.jpg`
- `ocr_region_crops_hybrid/table/*.jpg`
- the official metric notebook or `kaggle_metric.py` for `_normalize_text` and `_levenshtein`

Outputs:
- `formula_table_crop_predictions.jsonl`
- `formula_table_wrong_samples.json`


In [ ]:
INSTALL_DEPS = True

# Kaggle/local model inputs. Update these paths for your Kaggle input names.
BASE_MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1"
LORA_ADAPTER_DIR = "/kaggle/input/datasets/trankimhuu/qwen3vl-rukopys-lora-final-hard-type"
CROP_DATA_PARENT = "/kaggle/input/datasets/trankimhuu/ocr-region-crops/ocr_region_crops_hybrid"
LABELS_JSONL = "/kaggle/input/datasets/trankimhuu/ocr-region-crops/ocr_region_crops_hybrid/labels.jsonl"

# The official metric notebook exists in this repo locally. On Kaggle, add it as an input
# or provide a generated kaggle_metric.py and update this list.
METRIC_PATH_CANDIDATES = [
    "/kaggle/input/notebooks/dvoitekh/official-evaluation-metric-text-normalization/kaggle_metric.py",
]

OUTPUT_DIR = "/kaggle/working" if __import__("pathlib").Path("/kaggle/working").exists() else "artifacts/hard_samples_formula_table"
PREDICTIONS_JSONL = f"{OUTPUT_DIR}/formula_table_crop_predictions.jsonl"
PARTIAL_PREDICTIONS_PREFIX = f"{OUTPUT_DIR}/formula_table_crop_predictions_gpu"
WRONG_SAMPLES_JSON = f"{OUTPUT_DIR}/formula_table_wrong_samples.json"

GPU_IDS = [0, 1]

TARGET_TYPES = {"formula", "table"}
MAX_SAMPLES_PER_TYPE = None  # set a small int for smoke tests, for example 10

CROP_BATCH_SIZE = 8
CHECKPOINT_EVERY = 128
PROGRESS_EVERY = 128
MAX_PIXELS_CROP = 262_144
MAX_NEW_TOKENS_CROP = 1024
RESUME_PREDICTIONS = True
LOAD_LORA_CROP_PROMPTS = False

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "accelerate",
            "peft",
            "bitsandbytes",
            "qwen-vl-utils",
            "pandas==2.2.2",
            "pillow<12",
        ],
        [sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/transformers.git"],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd), flush=True)
        subprocess.check_call(cmd)


In [ ]:
import gc
import json
import logging
import multiprocessing as mp
import os
import re
import subprocess
import sys
import time
import warnings
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"


def suppress_transformers_noise():
    message = r".*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*"
    warnings.filterwarnings("ignore", message=message)
    logging.getLogger("transformers").setLevel(logging.ERROR)
    logging.getLogger("transformers.processing_utils").setLevel(logging.ERROR)
    try:
        from transformers.utils import logging as hf_logging

        hf_logging.set_verbosity_error()
    except Exception:
        pass


suppress_transformers_noise()
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR)


In [ ]:
BASE_MODEL_CANDIDATES = [BASE_MODEL_PATH, "Qwen/Qwen3-VL-8B-Instruct"]
LORA_CANDIDATES = [LORA_ADAPTER_DIR]


def find_model_id():
    for item in BASE_MODEL_CANDIDATES:
        if item.startswith("/") and Path(item).exists():
            return item
        if not item.startswith("/"):
            return item
    raise FileNotFoundError("No base model found. Add Qwen3-VL to Kaggle input or enable internet.")


def find_lora_dir():
    for item in LORA_CANDIDATES:
        p = Path(item)
        if (p / "adapter_config.json").exists():
            return p
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for p in input_root.rglob("adapter_config.json"):
            return p.parent
    raise FileNotFoundError("No LoRA adapter_config.json found. Add the fine-tuned adapter as a Kaggle input.")


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def iter_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def candidate_metric_paths():
    for value in METRIC_PATH_CANDIDATES:
        yield Path(value)
    input_root = Path("/kaggle/input")
    if input_root.exists():
        yield from input_root.rglob("kaggle_metric.py")
        yield from input_root.rglob("official-evaluation-metric-text-normalization.ipynb")


def metric_code_from_notebook(path):
    notebook = json.loads(Path(path).read_text(encoding="utf-8"))
    for cell in notebook.get("cells", []):
        source = "".join(cell.get("source", []))
        if "%%writefile kaggle_metric.py" in source and "def score(" in source:
            lines = source.splitlines()
            if lines and lines[0].startswith("%%writefile"):
                lines = lines[1:]
            return "\n".join(lines)
    raise RuntimeError(f"Cannot find kaggle_metric.py cell in {path}")


def load_metric_namespace():
    tried = []
    for path in candidate_metric_paths():
        tried.append(str(path))
        if not path.exists() or path.is_dir():
            continue
        if path.suffix.lower() == ".py":
            code_text = path.read_text(encoding="utf-8")
        elif path.suffix.lower() == ".ipynb":
            code_text = metric_code_from_notebook(path)
        else:
            continue
        namespace = {"__name__": "loaded_kaggle_metric"}
        exec(compile(code_text, str(path), "exec"), namespace)
        required = ["_normalize_text", "_levenshtein"]
        missing = [name for name in required if name not in namespace]
        if missing:
            raise RuntimeError(f"Metric file {path} is missing: {missing}")
        print("Loaded metric from", path)
        return namespace
    raise FileNotFoundError("No official metric found. Tried: " + ", ".join(tried[:20]))


model_id = find_model_id()
lora_dir = find_lora_dir()
metric_ns = load_metric_namespace()
normalize_text = metric_ns["_normalize_text"]
levenshtein = metric_ns["_levenshtein"]

print("Base model:", model_id)
print("LoRA:", lora_dir)


In [ ]:
VALID_TYPES = {"handwritten", "printed", "formula", "table", "annotation", "image", "graph"}

SOURCE_HINTS = {
    "dictation": "Ukrainian dictation handwriting. Do not complete from canonical text; read only visible characters.",
    "archive": "Historical Ukrainian/Cyrillic document. Preserve old spelling; do not modernize.",
    "school": "School homework. It may contain corrections, teacher marks, formulas, and mixed handwriting/print.",
    "university": "University exam/coursework. It may contain formulas, tables, chemistry notation, and technical symbols.",
}
DEFAULT_SOURCE_HINT = "Read only visible characters from this crop."

SPECIAL_TEXT_MARKER_RULES = (
    "Use [illegible] only for unreadable words inside an otherwise legible text region. "
    "Use ~~word~~ for visible strikethrough and ~~old~~{new} for visible correction."
)

STAGE_B_GUARDRAILS = (
    "The final transcription must be supported by the crop. "
    "Do not complete missing words from source hint, language prior, or canonical dictation text. "
    "Do not translate, correct grammar, normalize spelling, expand abbreviations, summarize, "
    "or infer hidden/missing text. No JSON, no Markdown, no explanation."
)

CROP_PROMPTS = {
    "formula": (
        "Read this standalone math, logic, vector, matrix, determinant, set/relation, statistics, physics, "
        "or chemistry expression exactly as written. Return only formula text, using LaTeX when it is the "
        "clearest representation and plain Unicode when it better matches the handwriting. Preserve visible "
        "symbols, indices, superscripts, subscripts, arrows, fractions, matrix/determinant structure, punctuation, "
        "numbering, and strikethrough/correction markers. Do not solve, simplify, normalize, explain, or convert "
        "old notation into a different style."
    ),
    "table": (
        "Read this table region exactly. Return only pipe-separated table text. Use one output line per visual row "
        "and | between cells. Preserve empty cells with empty fields, for example A||C. Preserve row order, "
        "column order, multi-word cell text, wrapped cell text, numbers, units, punctuation, dashes, and visible "
        "spelling mistakes. Do not infer missing cells, do not rebalance columns, do not summarize, and do not explain."
    ),
    "default": "Transcribe the visible content exactly. Preserve punctuation, corrections, and visible spacing. Return only text.",
}


def normalize_type(value):
    value = str(value or "handwritten").strip().lower()
    return value if value in VALID_TYPES else "handwritten"


def get_source_hint(source):
    source = str(source or "").strip().lower()
    return SOURCE_HINTS.get(source, DEFAULT_SOURCE_HINT)


def build_crop_prompt(rtype, source=None):
    rtype = normalize_type(rtype)
    type_prompt = CROP_PROMPTS.get(rtype, CROP_PROMPTS["default"])
    return "\n".join([get_source_hint(source), type_prompt, SPECIAL_TEXT_MARKER_RULES, STAGE_B_GUARDRAILS])


def clean_crop_text(text):
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    text = re.sub(r"^(text|transcription|answer)\s*:\s*", "", text, flags=re.I).strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
        text = text[1:-1].strip()
    if text.startswith("[") or text.startswith("{"):
        try:
            obj = json.loads(text)
            if isinstance(obj, dict) and "text" in obj:
                text = str(obj["text"])
            else:
                return ""
        except Exception:
            return ""
    return text[:500]


def load_prompt_config(lora_dir):
    global CROP_PROMPTS, MAX_PIXELS_CROP
    cfg_path = Path(lora_dir) / "rukopys_prompt_config.json"
    if not cfg_path.exists():
        return
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    if LOAD_LORA_CROP_PROMPTS:
        CROP_PROMPTS.update(cfg.get("crop_prompts", {}))
    MAX_PIXELS_CROP = int(cfg.get("max_pixels_crop", MAX_PIXELS_CROP))


load_prompt_config(lora_dir)


In [ ]:
def resolve_labels_path():
    path = Path(LABELS_JSONL)
    if path.exists():
        return path
    parent_candidate = Path(CROP_DATA_PARENT) / "ocr_region_crops_hybrid" / "labels.jsonl"
    if parent_candidate.exists():
        return parent_candidate
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for p in input_root.rglob("labels.jsonl"):
            if p.parent.name == "ocr_region_crops_hybrid":
                return p
    raise FileNotFoundError(f"Cannot find labels.jsonl from LABELS_JSONL={LABELS_JSONL}")


def resolve_crop_path(label, labels_path):
    raw = Path(str(label["crop_path"]))
    candidates = []
    if raw.is_absolute():
        candidates.append(raw)
    candidates.extend([
        Path(CROP_DATA_PARENT) / raw,
        labels_path.parent.parent / raw,
        labels_path.parent / raw.name,
        raw,
    ])
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]


labels_path = resolve_labels_path()
labels = read_jsonl(labels_path)
records = []
missing_crops = []
for row in labels:
    rtype = normalize_type(row.get("type"))
    if rtype not in TARGET_TYPES:
        continue
    crop_path = resolve_crop_path(row, labels_path)
    item = dict(row)
    item["type"] = rtype
    item["crop_path_abs"] = str(crop_path)
    item["sample_name"] = Path(str(row.get("crop_path", crop_path))).name
    if crop_path.exists():
        records.append(item)
    else:
        missing_crops.append(item)

if MAX_SAMPLES_PER_TYPE is not None:
    limited = []
    for rtype in sorted(TARGET_TYPES):
        limited.extend([r for r in records if r["type"] == rtype][: int(MAX_SAMPLES_PER_TYPE)])
    records = limited

counts = pd.Series([r["type"] for r in records]).value_counts().to_dict()
print("Labels:", labels_path)
print("Records to OCR:", len(records), counts)
print("Missing crop files:", len(missing_crops))
if missing_crops[:3]:
    print("Missing examples:", [x.get("crop_path") for x in missing_crops[:3]])
assert records, "No formula/table crop records found. Check CROP_DATA_PARENT and LABELS_JSONL."


In [ ]:
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig


def configure_processor_for_generation(processor):
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = "left"
    return processor


def load_qwen_model(device):
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    base = AutoModelForImageTextToText.from_pretrained(
        model_id,
        device_map={"": device},
        quantization_config=quantization_config,
        dtype=torch.float16,
        trust_remote_code=True,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    model = PeftModel.from_pretrained(base, str(lora_dir))
    model.eval()
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    processor = configure_processor_for_generation(processor)
    if processor.tokenizer.pad_token_id is not None:
        model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    return model, processor


def apply_chat_template(processor, messages):
    candidates = [
        {"tokenize": False, "add_generation_prompt": True, "template_kwargs": {"enable_thinking": False}},
        {"tokenize": False, "add_generation_prompt": True, "processor_kwargs": {"enable_thinking": False}},
        {"tokenize": False, "add_generation_prompt": True, "enable_thinking": False},
        {"tokenize": False, "add_generation_prompt": True},
    ]
    for kwargs in candidates:
        try:
            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "ignore",
                    message=r".*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*",
                )
                return processor.apply_chat_template(messages, **kwargs)
        except TypeError:
            continue
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_batch(model, processor, messages_batch, device, max_new_tokens):
    processor.tokenizer.padding_side = "left"
    texts = [apply_chat_template(processor, m) for m in messages_batch]
    image_inputs, video_inputs = process_vision_info(messages_batch)
    try:
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            text_kwargs={"padding": True, "return_tensors": "pt"},
            images_kwargs={"return_tensors": "pt"},
            videos_kwargs={"return_tensors": "pt"},
        )
    except TypeError:
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
    inputs = inputs.to(device)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
        )
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    decoded = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    del inputs, out, trimmed
    torch.cuda.empty_cache()
    return decoded


def resize_to_pixel_budget(img, max_pixels):
    w, h = img.size
    total = max(1, w * h)
    if total <= max_pixels:
        return img
    scale = (max_pixels / total) ** 0.5
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    return img.resize((new_w, new_h), Image.Resampling.LANCZOS)


def load_crop_image(crop_path):
    with Image.open(crop_path) as img:
        return resize_to_pixel_budget(img.convert("RGB"), MAX_PIXELS_CROP)


def crop_message(record):
    prompt = build_crop_prompt(record["type"], source=record.get("source"))
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": load_crop_image(record["crop_path_abs"])},
                {"type": "text", "text": prompt},
            ],
        }
    ]


print("Qwen helper functions ready. Each GPU worker will load its own model copy.")


In [ ]:
def load_existing_predictions(path):
    path = Path(path)
    if not RESUME_PREDICTIONS or not path.exists():
        return {}
    existing = {}
    for row in iter_jsonl(path):
        existing[row["crop_path"]] = row
    return existing


def append_jsonl_rows(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8", newline="\n") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n")


def write_jsonl_rows(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n")


def partial_predictions_path(gpu_id):
    return Path(f"{PARTIAL_PREDICTIONS_PREFIX}{gpu_id}.jsonl")


def maybe_print_gpu_inventory():
    try:
        out = subprocess.check_output(["nvidia-smi", "-L"], text=True).strip()
        print(out, flush=True)
    except Exception as exc:
        print("Could not read nvidia-smi -L:", exc, flush=True)


def make_prediction_row(record, pred_text, error=None):
    row = {
        "crop_path": record["crop_path"],
        "sample_name": record["sample_name"],
        "type": record["type"],
        "prediction": pred_text,
        "label_text": str(record.get("text") or ""),
        "source": record.get("source"),
        "source_file_name": record.get("source_file_name"),
        "row_index": record.get("row_index"),
        "region_index": record.get("region_index"),
    }
    if error:
        row["error"] = str(error)
    return row


def worker_process(gpu_id, worker_records, partial_path):
    suppress_transformers_noise()
    device = f"cuda:{gpu_id}"
    torch.cuda.set_device(gpu_id)
    print(f"[GPU {gpu_id}] loading Qwen+LoRA for {len(worker_records)} assigned crops", flush=True)
    qwen_model, processor = load_qwen_model(device)
    print(f"[GPU {gpu_id}] model loaded", flush=True)

    existing = load_existing_predictions(partial_path)
    todo = [record for record in worker_records if record["crop_path"] not in existing]
    print(f"[GPU {gpu_id}] resume partial={len(existing)} todo={len(todo)}", flush=True)

    start_time = time.time()
    new_done = 0
    for start in range(0, len(todo), CROP_BATCH_SIZE):
        batch = todo[start:start + CROP_BATCH_SIZE]
        messages = [crop_message(record) for record in batch]
        rows = []
        try:
            outs = generate_batch(qwen_model, processor, messages, device, MAX_NEW_TOKENS_CROP)
            for record, out in zip(batch, outs):
                rows.append(make_prediction_row(record, clean_crop_text(out)))
        except Exception as exc:
            print(f"[GPU {gpu_id}] OCR batch failed at local_start={start}: {exc}", flush=True)
            torch.cuda.empty_cache()
            gc.collect()
            for record in batch:
                rows.append(make_prediction_row(record, "", error=exc))

        append_jsonl_rows(partial_path, rows)
        new_done += len(rows)
        total_done = len(existing) + new_done
        if total_done % PROGRESS_EVERY == 0 or total_done == len(worker_records):
            elapsed = max(1e-6, time.time() - start_time)
            print(
                f"[GPU {gpu_id}] done={total_done}/{len(worker_records)} "
                f"new_speed={new_done / elapsed:.2f} crops/s last={rows[-1]['sample_name']}",
                flush=True,
            )
        if new_done % CHECKPOINT_EVERY == 0:
            torch.cuda.empty_cache()
            gc.collect()

    print(f"[GPU {gpu_id}] saved partial {partial_path}", flush=True)


def merge_prediction_files(output_path, partial_paths):
    merged = load_existing_predictions(output_path)
    for path in partial_paths:
        for row in load_existing_predictions(path).values():
            merged[row["crop_path"]] = row
    ordered = [merged[record["crop_path"]] for record in records if record["crop_path"] in merged]
    write_jsonl_rows(output_path, ordered)
    return ordered


mp.set_start_method("fork", force=True)
maybe_print_gpu_inventory()

for gpu_id in GPU_IDS:
    if gpu_id < 0:
        raise ValueError(f"Invalid GPU id: {gpu_id}")

prediction_path = Path(PREDICTIONS_JSONL)
partial_paths = [partial_predictions_path(gpu_id) for gpu_id in GPU_IDS]

if not RESUME_PREDICTIONS:
    for path in [prediction_path, *partial_paths]:
        if path.exists():
            path.unlink()

existing = load_existing_predictions(prediction_path)
for path in partial_paths:
    existing.update(load_existing_predictions(path))

todo_records = [record for record in records if record["crop_path"] not in existing]
print("Total records:", len(records), "already done:", len(existing), "todo:", len(todo_records), "gpu_ids:", GPU_IDS)

chunks = [todo_records[i::len(GPU_IDS)] for i in range(len(GPU_IDS))]
processes = []
for gpu_id, chunk in zip(GPU_IDS, chunks):
    partial_path = partial_predictions_path(gpu_id)
    if not chunk:
        print(f"[GPU {gpu_id}] no new crops assigned", flush=True)
        continue
    p = mp.Process(target=worker_process, args=(gpu_id, chunk, partial_path))
    p.start()
    processes.append(p)

for p in processes:
    p.join()

bad_exitcodes = [p.exitcode for p in processes if p.exitcode not in (0, None)]
if bad_exitcodes:
    raise RuntimeError(f"One or more GPU workers failed with exit codes: {bad_exitcodes}")

merged_rows = merge_prediction_files(prediction_path, partial_paths)
print("Merged predictions:", len(merged_rows), "->", prediction_path)


In [ ]:
pred_rows = read_jsonl(PREDICTIONS_JSONL)
pred_by_path = {row["crop_path"]: row for row in pred_rows}

wrong_samples = []
missing_predictions = []
correct_count = 0
cer_values = []

for record in records:
    pred = pred_by_path.get(record["crop_path"])
    if pred is None:
        missing_predictions.append(record["crop_path"])
        continue
    rtype = record["type"]
    label_text = str(record.get("text") or "")
    pred_text = str(pred.get("prediction") or "")
    label_norm = normalize_text(label_text, rtype)
    pred_norm = normalize_text(pred_text, rtype)
    distance = levenshtein(pred_norm, label_norm)
    cer = distance / max(len(label_norm), 1)
    cer_values.append(cer)
    if cer > 0:
        wrong_samples.append({
            "sample_name": record["sample_name"],
            "crop_path": record["crop_path"],
            "type": rtype,
            "cer": cer,
            "edit_distance": distance,
            "label_text": label_text,
            "prediction": pred_text,
            "label_norm": label_norm,
            "prediction_norm": pred_norm,
            "source": record.get("source"),
            "source_file_name": record.get("source_file_name"),
            "row_index": record.get("row_index"),
            "region_index": record.get("region_index"),
            "bbox": record.get("bbox"),
            "crop_bbox": record.get("crop_bbox"),
        })
    else:
        correct_count += 1

wrong_samples = sorted(wrong_samples, key=lambda x: (x["type"], -x["cer"], x["sample_name"]))
summary = {
    "labels_jsonl": str(labels_path),
    "predictions_jsonl": str(Path(PREDICTIONS_JSONL)),
    "target_types": sorted(TARGET_TYPES),
    "total_records": len(records),
    "predicted_records": len(pred_by_path),
    "missing_predictions": len(missing_predictions),
    "correct_count": correct_count,
    "wrong_count": len(wrong_samples),
    "avg_cer": float(sum(cer_values) / len(cer_values)) if cer_values else None,
    "wrong_by_type": pd.Series([x["type"] for x in wrong_samples]).value_counts().to_dict() if wrong_samples else {},
}

output = {
    "summary": summary,
    "wrong_sample_names": [x["sample_name"] for x in wrong_samples],
    "wrong_samples": wrong_samples,
    "missing_predictions": missing_predictions,
}
Path(WRONG_SAMPLES_JSON).parent.mkdir(parents=True, exist_ok=True)
Path(WRONG_SAMPLES_JSON).write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Wrong samples JSON:", WRONG_SAMPLES_JSON)

pd.DataFrame(wrong_samples).head(30)
